In [7]:
import os
import sys
from pathlib import Path
import yaml


def validate_quiz_yaml(file_path: Path) -> tuple[bool, list[str]]:
    """Validates a Quiz YAML file against Skill Map schema and checks for logical bugs."""
    errors = []

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = yaml.safe_load(f)
    except yaml.YAMLError as e:
        return False, [f"YAML Syntax Error: {e}"]
    except Exception as e:
        return False, [f"File Read Error: {e}"]

    if not isinstance(data, dict):
        return False, ["Root structure must be a dictionary."]

    # Root key validation
    if "quiz_title" not in data or not isinstance(data["quiz_title"], str):
        errors.append("Missing or invalid root key: 'quiz_title'.")

    questions = data.get("questions")
    if not isinstance(questions, list):
        errors.append("Missing or invalid root key: 'questions' (must be a list).")
        return False, errors

    # Question-level validation
    for index, q in enumerate(questions, start=1):
        if not isinstance(q, dict):
            errors.append(f"Item {index} in 'questions' is not a valid map/dictionary.")
            continue

        q_num = q.get("question_number", index)
        question_str = q.get("question", f"<Question #{q_num}>")

        options = q.get("options")
        correct_answer = q.get("correct_answer")

        # 1. Validate 'options' field
        if not isinstance(options, list):
            errors.append(f"Q{q_num} ('{question_str}'): 'options' must be a list.")
            options = []
        elif len(options) != 4:
            errors.append(
                f"Q{q_num} ('{question_str}'): Expected exactly 4 options, found {len(options)}."
            )

        # 2. Validate 'correct_answer' presence & match
        if correct_answer is None:
            errors.append(f"Q{q_num} ('{question_str}'): Missing 'correct_answer'.")
        elif options and correct_answer not in options:
            errors.append(
                f"BUG DETECTED in Q{q_num} ('{question_str}'):\n"
                f"   `correct_answer`: \"{correct_answer}\"\n"
                f"   `options`: {options}"
            )

        # 3. Check for duplicate options
        if len(options) != len(set(options)):
            errors.append(f"Q{q_num} ('{question_str}'): Contains duplicate options.")

        # 4. Check for audio track URL string format
        audio_url = q.get("audio_track_url")
        if not audio_url or not isinstance(audio_url, str):
            errors.append(f"Q{q_num} ('{question_str}'): Missing or invalid 'audio_track_url'.")

    return len(errors) == 0, errors


def collect_yaml_files(input_paths: list[str]) -> list[Path]:
    """Recursively finds all .yaml/.yml files within given directory or file paths."""
    yaml_files = []
    for path_str in input_paths:
        p = Path(path_str)
        if p.is_dir():
            # Recursively search directories for .yaml and .yml extensions
            yaml_files.extend(p.rglob("*.yaml"))
            yaml_files.extend(p.rglob("*.yml"))
        elif p.is_file() and p.suffix.lower() in [".yaml", ".yml"]:
            yaml_files.append(p)
        elif not p.exists():
            print(f"⚠️ Warning: Path does not exist: {path_str}")

    return sorted(list(set(yaml_files)))


def run_validator(input_paths: list[str]):
    yaml_files = collect_yaml_files(input_paths)

    if not yaml_files:
        print("No YAML files found in the specified path(s).")
        return

    print(f"Found {len(yaml_files)} YAML file(s) to scan.\n" + "=" * 50)

    total_bugs = 0
    failed_files = 0

    for path in yaml_files:
        # Skip structure.yaml - this is a system YAML to store the folder sturcture in the app
        if path.name.lower() == "structure.yaml":
            print(f"⏭️ SKIP: {path}")
            continue

        is_valid, errors = validate_quiz_yaml(path)
        if is_valid:
            print(f"✅ PASS: {path}")
        else:
            failed_files += 1
            total_bugs += len(errors)
            print(f"❌ FAIL: {path}")
            for err in errors:
                print(f"   • {err}")
            print("-" * 50)

    print("\n" + "=" * 50)
    print(f"Scan complete. {len(yaml_files) - failed_files}/{len(yaml_files)} passed.")
    if failed_files > 0:
        print(f"Found {total_bugs} total issue(s) across {failed_files} file(s).")
        sys.exit(1)


if __name__ == "__main__":
    run_validator([r"G:\My Drive\Study\Skillmap\spanish\tests"])

Found 8 YAML file(s) to scan.
✅ PASS: G:\My Drive\Study\Skillmap\spanish\tests\spanish-04-spanish-travel-survival-01.yaml
✅ PASS: G:\My Drive\Study\Skillmap\spanish\tests\spanish-test-02.yaml
✅ PASS: G:\My Drive\Study\Skillmap\spanish\tests\spanish-test-30-core-verbs.yaml
✅ PASS: G:\My Drive\Study\Skillmap\spanish\tests\spanish-test-numbers-advanced.yaml
✅ PASS: G:\My Drive\Study\Skillmap\spanish\tests\spanish-test-prices-money.yaml
✅ PASS: G:\My Drive\Study\Skillmap\spanish\tests\spanish-test-time-dates.yaml
⏭️ SKIP: G:\My Drive\Study\Skillmap\spanish\tests\structure.yaml
✅ PASS: G:\My Drive\Study\Skillmap\spanish\tests\test-01.yaml

Scan complete. 8/8 passed.
